# Cuál es el problema?

Abandono de clientes (Churn)

Trabajamos con una empresa de telecomunicaciones que quiere ***predecir cuales clientes van a abandonar el servicio*** (churn) para poder tomar acciones preventivas.

Por qué es importante?

* Conseguir un cliente nuevo representa entre 5x y 25x más que retener uno existente.
* Si podemos predecir quien se va, podemos ofrecerle un descuento o llamarlo antes de que cancele.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer # Para transformar columnas específicas
from sklearn.preprocessing import OneHotEncoder # Para escalar y codificar variables
from sklearn.linear_model import LogisticRegression # Modelo de regresión logística

# Metricas de evaluación para la clasificación

from sklearn.metrics import (
    confusion_matrix, # Matriz de confusión
    classification_report, # reporte con precisión, recall y f1-score
    accuracy_score, # Porcentaje de predicciones correctas
)

import matplotlib.pyplot as plt

In [2]:
#Cargar el archivo CSV en un DataFrame (tabla de datos  )
df = pd.read_csv("abandono_clientes.csv")
# Mostrar las primeras filas del DataFrame para verificar que se cargó correctamente
print("primeras 5 filas del DataSet:")
df.head()

primeras 5 filas del DataSet:


,antiguedad_meses,factura_mensual,llamadas_soporte,contrato,satisfaccion,abandono
0,52,61.18,0,Dos_anios,3,0
1,15,54.89,1,Mensual,1,1
2,72,112.95,3,Anual,2,0
3,61,103.06,0,Mensual,1,1
4,21,116.50,2,Mensual,1,1


In [3]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

Filas: 800, Columnas: 6

Tipos de datos:
  antiguedad_meses     -> int64
  factura_mensual      -> float64
  llamadas_soporte     -> int64
  contrato             -> str
  satisfaccion         -> int64
  abandono             -> int64

No hay valores faltantes en el dataset.


aqui viene algo

In [5]:
#Comporar el promedio de cada variable 

comparacion = df.groupby("abandono")[["antiguedad_meses","factura_mensual",
                                      "llamadas_soporte","satisfaccion"]].mean().round(2)
comparacion.index=["Permanece (0)","Abandona (1)"]
print("Promedio de variables numerica segun abandono:")
comparacion

Promedio de variables numerica segun abandono:


,antiguedad_meses,factura_mensual,llamadas_soporte,satisfaccion
Permanece (0),39.43,67.02,1.88,3.14
Abandona (1),28.41,77.27,2.28,2.67


In [6]:
# Analisi del tipo contrato vs el abandono
tabla_contrato = pd.crosstab(
    df["contrato"], 
    df["abandono"], 
    normalize="index"
    ) * 100
tabla_contrato.columns = ["Permanece (%)", "Abandona (%)"]
print("porcentaje de abandono pro tipo de contrato:")
comparacion

porcentaje de abandono pro tipo de contrato:


,antiguedad_meses,factura_mensual,llamadas_soporte,satisfaccion
Permanece (0),39.43,67.02,1.88,3.14
Abandona (1),28.41,77.27,2.28,2.67


# preparar los daros para el modelo

1. separar variables predictoras (x) de la variable objetivo
2. Entrenar los datos con (80%) del dataset y probar con el (20%) del dataset.

In [17]:
# paso 1: separar x (variables predictoras) e y (variable objetivo)
X = df[["antiguedad_meses", "factura_mensual", "llamadas_soporte", "satisfaccion", "contrato"]]
y = df["abandono"]

# paso 2: dividir el conjunto de entrenamiento (80%) y prueba (20% )
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, #20% para prueba
        random_state=42,  # Semilla para reproducibilidad
    stratify=y  
)

print(f"Datos de entrenamiento: {X_train.shape[0]} clientes")
print(f"Datos de prueba: {X_test.shape[0]} clientes")
print("")
print(f"Proporción de abandono en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"Proporción de abandono en prueba: {y_test.mean()*100:.1f}%")
print()
print("Las proporciones son similares gracias al parametro 'stratify=y' en train_test_split")


Datos de entrenamiento: 640 clientes
Datos de prueba: 160 clientes

Proporción de abandono en entrenamiento: 35.3%
Proporción de abandono en prueba: 35.6%

Las proporciones son similares gracias al parametro 'stratify=y' en train_test_split


# Construir  el pipeline y entrenar el modelo

1. Calula una puntuación lineal
2.Pasa por una puntación que la convierte en probabilidad entre 0 y 1
3. SI la probabilidad es mayor a 0.5  predice la clase 1 (abandona): si no, predice 0 (permanece)

In [20]:
from sklearn.pipeline import Pipeline

numeric_features = ["antiguedad_meses", "factura_mensual",
                    "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]


preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)


pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

# Entrenar el modelo
pipe.fit(X_train, y_train)

print("Modelo entrenado con éxito.")

Modelo entrenado con éxito.


# Hacer Predcciones

Usamos el modelo entrenado para predecir el abandono en los datos de prueba

In [21]:
# Generar predicciones con los datos de prueba
y_pred = pipe.predict(X_test)

# Mostrar las primeras 10 predicciones vs valores reales
comparacion_pred = pd.DataFrame({
    "Real":      y_test.values[:10],
    "Prediccion": y_pred[:10],
    "Correcto":  ["Si" if r == p else "NO" for r, p in zip(y_test.values[:10], y_pred[:10])]
})

print("Primeras 10 predicciones vs valores reales:")
print()
comparacion_pred

Primeras 10 predicciones vs valores reales:



,Real,Prediccion,Correcto
0,1,1,Si
1,0,0,Si
2,0,0,Si
3,0,0,Si
4,0,0,Si
5,0,0,Si
6,0,0,Si
7,0,0,Si
8,0,0,Si
9,0,0,Si


# Evaluacion del modelo: Exactitu/precision (Acurracy)

cuando hablamos del porcentaje de precision nos referimos al % de predicciones que fueron correctas

Accuracy = Predicciones Correctas / Total de predicciones

Ejemplo: si el 95% de clientes **No** abandona, un modelo que siempre diga "No abandona" tendría un 95% de presicion, pero jamas detectaria a un cliente en riesgo. sería inutil.

In [22]:
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy (exactitud): {acc:.4f} ({acc*100:.1f}%%)")
print()
print(f"esto significa que el modelo acerto {int(acc*len(y_test))} de {len(y_test)} predicciones")


Accuracy (exactitud): 0.7625 (76.2%%)

esto significa que el modelo acerto 122 de 160 predicciones
